# Multi-Modal LLM using EdenAI for image reasoning

In this notebook, we show how to use MultiModal LLM class for image understanding/reasoning.


In [ ]:
%pip install llama-index-multi-modal-llms-edenai


Usage:   
  /home/redha/.cache/pypoetry/virtualenvs/llama-index-kg8bGAY7-py3.10/bin/python -m pip <command> [options]

no such option: -e
Note: you may need to restart the kernel to use updated packages.


## Load and initialize EdenAI 

In [1]:
import os


EDENAI_API_KEY = ""  # Your edenai API token here
os.environ["EDENAI_API_KEY"] = EDENAI_API_KEY


## Download Images and Load Images locally

In [4]:
import os
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
from llama_index.core.multi_modal_llms.generic_utils import load_image_urls
from llama_index.core.schema import ImageDocument

if not os.path.exists("test_images"):
    os.makedirs("test_images")

image_urls = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRFU7U2h0umyF0P6E_yhTX45sGgPEQAbGaJ4g&s",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/481px-Cat03.jpg",
]

image_documents = [
    ImageDocument(image_url=url)
    for url in image_urls
]


## Provide various prompts to test different Multi Modal LLMs

In [5]:
from llama_index.multi_modal_llms.edenai import EdenaiMultiModal

prompts = [
    "what is shown in this image?",
    "what animal is in the picture?",
]


## Generate Image Reasoning from different LLMs with different prompts for different images

In [ ]:
res = []
llm_model = "google/gemini-1.5-flash"
for prompt_idx, prompt in enumerate(prompts):
    for image_idx, image_doc in enumerate(image_documents):
        try:
            multi_modal_llm = EdenaiMultiModal(
                model_name=llm_model,
                top_p=0.9,
                api_key=EDENAI_API_KEY,
            )

            mm_resp = multi_modal_llm.complete(
                prompt=prompt,
                image_documents=[image_doc],
            )
            print(mm_resp.raw)
        except Exception as e:
            print(
                f"Error with LLM model inference with prompt {prompt}, image {image_idx}, and MM model {llm_model}"
            )
            print("Inference Failed due to: ", e)
            continue
        res.append(
            {
                "model": llm_model,
                "prompt": prompt,
                "response": mm_resp,
            }
        )

## Display Sampled Responses from Multi-Modal LLMs 

### Using complete 

In [17]:
from IPython.display import display
import pandas as pd

pd.options.display.max_colwidth = None
df = pd.DataFrame(res)
display(df[:5])

,model,prompt,response,image
0,google/gemini-1.5-flash,what is shown in this image?,That's a cute image of a plush panda bear sitting in grass and eating a stalk of bamboo. The background features bamboo stalks and a partly cloudy sky. The panda appears to be a stuffed toy or a cartoon rendering.\n,None
1,google/gemini-1.5-flash,what is shown in this image?,"That's a close-up photo of a ginger cat. The cat is looking slightly away from the camera, and its fur is a rich, light orange color. The background is blurred, making the cat the clear focus of the image. A reddish-colored object (possibly a rope or hose) is partially visible in the out-of-focus background.\n",None
2,google/gemini-1.5-flash,what animal is in the picture?,"That's a panda. More specifically, it appears to be a stylized or cartoonish representation of a giant panda ( *Ailuropoda melanoleuca*).\n",None
3,google/gemini-1.5-flash,what animal is in the picture?,"That's a ginger cat. More specifically, it appears to be a domestic shorthair cat with a ginger or orange coat.\n",None


### Using async complete

In [11]:
resp = await multi_modal_llm.acomplete(
    prompt="tell me about this image",
    image_documents=[image_documents[0]],
)

In [12]:
print(resp)

That's a cute image! It depicts a plush panda bear sitting amidst bamboo stalks. 


Here's a breakdown of the image:

* **The Panda:** The panda is the central focus, large and fluffy-looking.  It's holding a stalk of bamboo in its paws, as if about to eat it.  The panda's expression is friendly and somewhat comical. The plush toy style gives it a soft, cuddly appearance.

* **The Setting:** The background features several stalks of bamboo, suggesting the panda's natural habitat. The sky is a light blue, implying a daytime setting. There's also green grass at the bottom of the image.  The overall scene is peaceful and idyllic.

* **Overall Impression:** The image is designed to be cheerful and appealing, likely intended for children or anyone who enjoys cute animals. The contrast between the soft panda and the crisp bamboo creates a visually pleasing effect.



### Using chat 


In [ ]:
from llama_index.multi_modal_llms.edenai import EdenaiMultiModal
from llama_index.core.llms import (
    ChatMessage,
    ImageBlock,
    TextBlock,
    MessageRole,
)

llm_model = "openai/gpt-4o"
results = []
for prompt_idx, prompt in enumerate(prompts):
    for image_idx, image_doc in enumerate(image_documents):
        try:
            messages = [
                ChatMessage(
                    role=MessageRole.USER,
                    blocks=[
                        TextBlock(text=prompt),
                        ImageBlock(image=None, path=None,url=image_doc.image_url)  
                    ]
                )
            ]

            multi_modal_llm = EdenaiMultiModal(
                model_name=llm_model,
                top_p=0.9,
                api_key=EDENAI_API_KEY,
            )

            mm_resp = multi_modal_llm.chat(messages=messages)

        except Exception as e:
            print(
                f"Error with LLM model inference: prompt '{prompt}', image {image_idx}, model '{llm_model}'."
            )
            print("Inference Failed due to: ", e)
            continue

        results.append(
            {
                "model": llm_model,
                "prompt": prompt,
                "response": mm_resp,
            }
        )


In [16]:
from IPython.display import display
import pandas as pd

pd.options.display.max_colwidth = None
df = pd.DataFrame(results)
display(df[:5])

,model,prompt,response,image
0,openai/gpt-4o,what is shown in this image?,assistant: The image shows an animated panda sitting and chewing on a piece of bamboo. The background features bamboo stalks and a grassy area. The panda appears to be smiling.,None
1,openai/gpt-4o,what is shown in this image?,"assistant: The image shows a ginger tabby cat, looking directly at the camera. The cat has orange fur with darker stripes and striking yellow eyes.",None
2,openai/gpt-4o,what animal is in the picture?,assistant: The image shows a cartoon panda.,None
3,openai/gpt-4o,what animal is in the picture?,assistant: The animal in the picture is a cat.,None
